In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import mean_absolute_error, mean_squared_error, classification_report

In [2]:
data_path = "../data/processed/data_processed.csv"
df = pd.read_csv(data_path, parse_dates=['date'])

In [3]:
df_sort = df.sort_values(by=['date', 'city']).reset_index(drop=True)
df_sort.head(24)

,city,latitude,longitude,date,temperature_2m_mean,rain_sum,precipitation_hours,weather_code,wind_speed_10m_mean,relative_humidity_2m_mean,...,dewpoint_2m_mean_lag_2,surface_pressure_mean_lag_2,cloudcover_mean_lag_2,wind_speed_10m_mean_lag_2,temperature_2m_mean_rolling_mean_3,relative_humidity_2m_mean_rolling_mean_3,dewpoint_2m_mean_rolling_mean_3,surface_pressure_mean_rolling_mean_3,cloudcover_mean_rolling_mean_3,wind_speed_10m_mean_rolling_mean_3
0,0,10.5417,107.2429,2014-01-08,-0.351351,0.125,1.0,1,0.510638,-0.773585,...,-0.993846,-0.197183,-0.412916,1.042553,-0.441441,-0.918239,-0.943932,-0.144422,-0.002712,6.950355e-01
1,1,16.0544,108.2022,2014-01-08,-1.486486,0.000,0.0,0,-0.212766,0.509434,...,-1.526154,0.820926,-1.465548,0.425532,-1.549550,0.246541,-1.275214,0.840599,-1.167218,2.198582e-01
2,2,10.9465,106.8340,2014-01-08,-0.162162,0.000,0.0,0,-0.617021,-1.007547,...,-0.899487,0.256204,-0.608187,-0.361702,-0.189189,-1.044182,-0.884103,0.316790,-0.261378,-3.971631e-01
3,3,21.0278,105.8342,2014-01-08,-2.324324,0.750,5.0,1,0.340426,-0.332075,...,-1.757949,0.393696,-0.325451,0.659574,-1.978604,-0.030189,-1.811282,0.622625,-0.100347,2.553191e-01
4,4,10.7769,106.7009,2014-01-08,-0.081081,0.000,0.0,0,-0.489362,-1.132075,...,-0.925128,0.156271,-0.533944,-0.276596,-0.162162,-1.073270,-0.923761,0.213280,-0.178998,-2.588652e-01
5,5,16.4637,107.5909,2014-01-08,-1.378378,0.000,0.0,0,-0.531915,0.596226,...,-1.417436,0.684105,-1.774727,0.659574,-1.342342,0.167296,-1.130598,0.736195,-1.343165,7.092199e-02
6,6,12.2585,109.0526,2014-01-08,-1.027027,0.000,0.0,0,1.212766,-0.033962,...,-1.368205,0.594232,-1.160437,1.255319,-1.027027,-0.402516,-1.155556,0.596691,-0.720061,1.127660e+00
7,7,10.5336,106.4110,2014-01-08,-0.243243,0.000,0.0,0,-0.489362,-0.728302,...,-0.779487,0.305835,-0.364099,-0.255319,-0.288288,-0.838994,-0.789060,0.358596,-0.087465,-2.836879e-01
8,8,15.5394,108.0191,2014-01-08,-1.432432,0.000,0.0,0,-0.702128,0.384906,...,-1.510769,0.179745,-1.433003,-0.297872,-1.423423,0.084277,-1.268718,0.197407,-0.971608,-4.680851e-01
9,9,21.0064,107.2925,2014-01-08,-2.540541,0.700,2.0,1,1.664894,-0.177358,...,-1.758974,0.799463,-0.101704,0.297872,-2.333333,0.394969,-1.891966,0.963783,-0.046784,6.187943e-01


In [4]:
def split_data(df, val_year = 2, test_year = 1):
    years = df['date'].dt.year.unique()
    print(years)
    val_years = years[-(val_year+test_year):-test_year]
    print(val_years)
    test_years = years[-test_year:]
    train_data = df[~df['date'].dt.year.isin(val_years)].reset_index(drop=True)
    val_data = df[df['date'].dt.year.isin(val_years)].reset_index(drop=True)
    test_data = df[df['date'].dt.year.isin(test_years)].reset_index(drop=True)
    return train_data, val_data, test_data

In [6]:
train_data, val_data, test_data = split_data(df_sort, val_year=2, test_year=1)

[2014 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024]
[2022 2023]


In [7]:
train_data.to_csv("../data/processed/train_data.csv", index=False)
val_data.to_csv("../data/processed/val_data.csv", index=False)
test_data.to_csv("../data/processed/test_data.csv", index=False)

In [8]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39372 entries, 0 to 39371
Data columns (total 58 columns):
 #   Column                                    Non-Null Count  Dtype         
---  ------                                    --------------  -----         
 0   city                                      39372 non-null  int64         
 1   latitude                                  39372 non-null  float64       
 2   longitude                                 39372 non-null  float64       
 3   date                                      39372 non-null  datetime64[ns]
 4   temperature_2m_mean                       39372 non-null  float64       
 5   rain_sum                                  39372 non-null  float64       
 6   precipitation_hours                       39372 non-null  float64       
 7   weather_code                              39372 non-null  int64         
 8   wind_speed_10m_mean                       39372 non-null  float64       
 9   relative_humidity_2m_mean   

In [9]:
val_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 58 columns):
 #   Column                                    Non-Null Count  Dtype         
---  ------                                    --------------  -----         
 0   city                                      8760 non-null   int64         
 1   latitude                                  8760 non-null   float64       
 2   longitude                                 8760 non-null   float64       
 3   date                                      8760 non-null   datetime64[ns]
 4   temperature_2m_mean                       8760 non-null   float64       
 5   rain_sum                                  8760 non-null   float64       
 6   precipitation_hours                       8760 non-null   float64       
 7   weather_code                              8760 non-null   int64         
 8   wind_speed_10m_mean                       8760 non-null   float64       
 9   relative_humidity_2m_mean     

In [10]:
test_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4392 entries, 0 to 4391
Data columns (total 58 columns):
 #   Column                                    Non-Null Count  Dtype         
---  ------                                    --------------  -----         
 0   city                                      4392 non-null   int64         
 1   latitude                                  4392 non-null   float64       
 2   longitude                                 4392 non-null   float64       
 3   date                                      4392 non-null   datetime64[ns]
 4   temperature_2m_mean                       4392 non-null   float64       
 5   rain_sum                                  4392 non-null   float64       
 6   precipitation_hours                       4392 non-null   float64       
 7   weather_code                              4392 non-null   int64         
 8   wind_speed_10m_mean                       4392 non-null   float64       
 9   relative_humidity_2m_mean     

### CONFIG

In [11]:
# Col to predict
target_column = 'rain_sum'
side_target = ['precipitation_hours', 'weather_code', "date"]

drop_cols = ['surface_pressure_mean', 'wind_speed_10m_mean','cloudcover_mean', 'temperature_2m_mean', 'relative_humidity_2m_mean', 'rain_sum','precipitation_hours', 'weather_code', 'date']

# Define training and testing sets
X_train = train_data.drop(columns=drop_cols)

X_test = val_data.drop(columns=drop_cols)



# METRICS


# XGBOOST

1. Model

In [12]:
import xgboost as xgb
import joblib

rf_params = {'n_estimators':1000,
    'learning_rate':0.05,
    'max_depth':6,
    'objective':'reg:squarederror',
    'enable_categorical':True,
    'device':'cuda' if torch.cuda.is_available() else 'cpu'}

models = {
    # --- NHÓM 1: DRIVERS (Dự báo trước) ---
    'surface_pressure_mean': xgb.XGBRegressor(**rf_params),
    'wind_speed_10m_mean': xgb.XGBRegressor(**rf_params),
    'cloudcover_mean': xgb.XGBRegressor(**rf_params),
    'temperature_2m_mean': xgb.XGBRegressor(**rf_params),
    
    # --- NHÓM 2: MOISTURE (Dự báo sau Drivers) ---
    'relative_humidity_2m_mean': xgb.XGBRegressor(**rf_params),
    
    # --- NHÓM 3: TARGETS (Dự báo cuối cùng) ---
    'rain_sum': xgb.XGBRegressor(**rf_params), 
    'precipitation_hours': xgb.XGBRegressor(**rf_params),
    'weather_code': xgb.XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    objective='multi:softmax',
    num_class=len(train_data['weather_code'].unique()),
    enable_categorical=True,
    device='cuda' if torch.cuda.is_available() else 'cpu'
) 
}

evaluation_results = {}
for name, model in models.items():
    model.fit(X_train, train_data[name])
    Y_pred = model.predict(X_test)
    y_true = val_data[name]
    if name == 'weather_code':
        report = classification_report(y_true, Y_pred, output_dict=True)
        evaluation_results[name] = report
        print(f"Weather Code - Classification Report:\n{classification_report(y_true, Y_pred)}")
    else:
        mae = mean_absolute_error(y_true=y_true, y_pred=Y_pred)
        mse = mean_squared_error(y_true=y_true, y_pred=Y_pred)
        evaluation_results[name] = {'MAE': mae, 'MSE': mse, 'RMSE': np.sqrt(mse)}
        print(f"{name} - MAE: {mae}, MSE: {mse}, RMSE: {np.sqrt(mse)}")
    joblib.dump(model, f"../models/xgboost/{name}_model.pkl")

C:\Users\buiti\AppData\Roaming\Python\Python312\site-packages\xgboost\core.py:729: UserWarning: [13:49:16] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


surface_pressure_mean - MAE: 0.024104521929764203, MSE: 0.002629829236041741, RMSE: 0.05128186069207845
wind_speed_10m_mean - MAE: 0.021709450888380968, MSE: 0.004630577602633919, RMSE: 0.06804834753786398
cloudcover_mean - MAE: 0.01661343264699156, MSE: 0.0006711561068682237, RMSE: 0.025906680738145976
temperature_2m_mean - MAE: 0.031744208452986575, MSE: 0.003126130920731747, RMSE: 0.055911813785028894
relative_humidity_2m_mean - MAE: 0.022857155549757407, MSE: 0.002604032558227527, RMSE: 0.051029722302081235
rain_sum - MAE: 0.3703426625902794, MSE: 3.31797105318491, RMSE: 1.8215298661248762
precipitation_hours - MAE: 0.14178758405114963, MSE: 0.052791576455222304, RMSE: 0.22976417574378802
Weather Code - Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1971
           1       1.00      1.00      1.00      6789

    accuracy                           1.00      8760
   macro avg       1.00      1.00      1.0

# RANDOM_FOREST

In [13]:
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

rf_params = {    'n_estimators':500,
    'max_depth':6,
    'min_samples_leaf' : 10,
    'min_samples_split' : 20,
    'max_features' : 'sqrt',
    'n_jobs':-1}

models = {
    # --- NHÓM 1: DRIVERS (Dự báo trước) ---
    'surface_pressure_mean': RandomForestRegressor(**rf_params),
    'wind_speed_10m_mean': RandomForestRegressor(**rf_params),
    'cloudcover_mean': RandomForestRegressor(**rf_params),
    'temperature_2m_mean': RandomForestRegressor(**rf_params),
    
    # --- NHÓM 2: MOISTURE (Dự báo sau Drivers) ---
    'relative_humidity_2m_mean': RandomForestRegressor(**rf_params),
    
    # --- NHÓM 3: TARGETS (Dự báo cuối cùng) ---
    'rain_sum': RandomForestRegressor(**rf_params), 
    'precipitation_hours': RandomForestRegressor(**rf_params),
    'weather_code': RandomForestClassifier(**rf_params, class_weight='balanced') 
}

evaluation_results = {}
for name, model in models.items():
    model.fit(X_train, train_data[name])
    Y_pred = model.predict(X_test)
    y_true = val_data[name]
    if name == 'weather_code':
        report = classification_report(y_true, Y_pred, output_dict=True)
        evaluation_results[name] = report
        print(f"Weather Code - Classification Report:\n{classification_report(y_true, Y_pred)}")
    else:
        mae = mean_absolute_error(y_true=y_true, y_pred=Y_pred)
        mse = mean_squared_error(y_true=y_true, y_pred=Y_pred)
        evaluation_results[name] = {'MAE': mae, 'MSE': mse, 'RMSE': np.sqrt(mse)}
        print(f"{name} - MAE: {mae}, MSE: {mse}, RMSE: {np.sqrt(mse)}")
    joblib.dump(model, f"../models/RandomForest/{name}_model.pkl")

surface_pressure_mean - MAE: 0.17663595014718847, MSE: 0.05916736248152108, RMSE: 0.24324342227801574
wind_speed_10m_mean - MAE: 0.29284789951945833, MSE: 0.15993685372714433, RMSE: 0.3999210593694015
cloudcover_mean - MAE: 0.28457559391474246, MSE: 0.1297028152532737, RMSE: 0.3601427706525201
temperature_2m_mean - MAE: 0.14626676024486956, MSE: 0.04307928524327471, RMSE: 0.20755549918822847
relative_humidity_2m_mean - MAE: 0.22914374844320717, MSE: 0.09833139002671439, RMSE: 0.3135783634543595
rain_sum - MAE: 3.2380865253948383, MSE: 37.005475332202096, RMSE: 6.083212583183502
precipitation_hours - MAE: 2.4291370661253024, MSE: 10.60954621780579, RMSE: 3.257229838038113
Weather Code - Classification Report:
              precision    recall  f1-score   support

           0       0.84      1.00      0.91      1971
           1       1.00      0.94      0.97      6789

    accuracy                           0.96      8760
   macro avg       0.92      0.97      0.94      8760
weighted a

# ROLLING PREDICT

LoadModel

In [22]:
models = {
    'surface_pressure_mean': joblib.load("../models/xgboost/surface_pressure_mean_model.pkl"),
    'wind_speed_10m_mean': joblib.load("../models/xgboost/wind_speed_10m_mean_model.pkl"),
    'cloudcover_mean': joblib.load("../models/xgboost/cloudcover_mean_model.pkl"),
    'temperature_2m_mean': joblib.load("../models/xgboost/temperature_2m_mean_model.pkl"),
    'relative_humidity_2m_mean': joblib.load("../models/xgboost/relative_humidity_2m_mean_model.pkl"),
    'rain_sum': joblib.load("../models/xgboost/rain_sum_model.pkl"),
    'precipitation_hours': joblib.load("../models/xgboost/precipitation_hours_model.pkl"),
    'weather_code': joblib.load("../models/xgboost/weather_code_model.pkl")
}

In [23]:
target_cols = ["rain_sum", "weather_code", "precipitation_hours"]
dynamic_cols = ['temperature_2m_mean', 
    'relative_humidity_2m_mean', 
    'dewpoint_2m_mean', 
    'surface_pressure_mean', 
    'cloudcover_mean', 
    'wind_speed_10m_mean']
predict_cols = ['surface_pressure_mean', 'wind_speed_10m_mean','cloudcover_mean', 'temperature_2m_mean', 'relative_humidity_2m_mean', 'rain_sum','precipitation_hours', 'weather_code']
lags_target = [1,2,3,7]
window_rolling_target = [3,7]
lags_dynamic = [1,2]
window_rolling_dynamic = [3]

In [24]:
def generate_lag_rolling_features(df_):
    input = df_.groupby('city').tail(1).copy()
    next_date = input['date'] + pd.Timedelta(days=1)
    input['date'] = next_date
    
    for col in predict_cols:
        input[col] = np.nan
    
    month_max = 12
    week_day_max = 7
    input['year'] = next_date.dt.year
    input['day_sin'] = np.sin(2 * np.pi * next_date.dt.day / next_date.dt.days_in_month)
    input['day_cos'] = np.cos(2 * np.pi * next_date.dt.day / next_date.dt.days_in_month)  
    input['dayofweek_sin'] = np.sin(2 * np.pi * next_date.dt.dayofweek / week_day_max)
    input['dayofweek_cos'] = np.cos(2 * np.pi * next_date.dt.dayofweek / week_day_max)
    input['month_sin'] = np.sin(2 * np.pi * next_date.dt.month / month_max)
    input['month_cos'] = np.cos(2 * np.pi * next_date.dt.month / month_max)
    input['quarter_sin'] = np.sin(2 * np.pi * next_date.dt.quarter / 4)
    input['quarter_cos'] = np.cos(2 * np.pi * next_date.dt.quarter / 4)
    temp_df = pd.concat([df_, input], ignore_index=True)
    temp_df = temp_df.sort_values(['date', 'city']).reset_index(drop=True)
    for col in target_cols:
        for lag in lags_target:
            temp_df[f"{col}_lag_{lag}"] = temp_df.groupby('city')[col].shift(lag)
        for window in window_rolling_target:
            temp_df[f"{col}_rolling_mean_{window}"] = temp_df.groupby('city')[col].transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
            
    for col in dynamic_cols:
        for lag in lags_dynamic:
            temp_df[f"{col}_lag_{lag}"] = temp_df.groupby('city')[col].shift(lag)
        for window in window_rolling_dynamic:
            temp_df[f"{col}_rolling_mean_{window}"] = temp_df.groupby('city')[col].transform(lambda x: x.rolling(window, min_periods=1).mean())
    final_input = temp_df.groupby('city').tail(1)
    final_input = final_input.drop(columns=drop_cols)
    return final_input, temp_df

In [25]:
def predict_weather(models, data, rolling_window=7): # hiện tại data nhận vào là train_data
    data_sorted = data.sort_values(by=['date', 'city']).reset_index(drop=True)
    predictions = {}
    data_past = data_sorted.groupby('city').tail(rolling_window+7).reset_index(drop=True)
    for i in range(rolling_window):
        input_row, data_past = generate_lag_rolling_features(data_past)
        predictions[f'day_{i+1}'] = {}
        for target, model in models.items():    
            pred = model.predict(input_row)
            predictions[f'day_{i+1}'][target] = pred
            data_past.loc[input_row.index, target] = pred
    return predictions, data_past

In [26]:
class RollingPredictor:
    def __init__(self, models, rolling_window=7):
        self.models = models
        self.rolling_window = rolling_window
        
    def predict(self, data):
        return predict_weather(self.models, data, self.rolling_window)

In [45]:
test_rolling = test_data.groupby('city').head(30).reset_index(drop=True)
test_rolling_predictions, test_rolling_data = predict_weather(models, test_rolling, rolling_window=1)
y_true = test_data.groupby('city').head(31).reset_index(drop=True)

In [46]:
y_true_rain = y_true.groupby('city')['rain_sum'].tail(1).values
y_true_precip_hours = y_true.groupby('city')['precipitation_hours'].tail(1).values
y_true_weather_code = y_true.groupby('city')['weather_code'].tail(1).values

In [47]:
y_pred_rain = []
for target, val in test_rolling_predictions.items():
    y_pred_rain.extend(val['rain_sum'])
y_pred_rain = np.array(y_pred_rain)

In [48]:
y_true_rain.shape, y_pred_rain.shape

((12,), (12,))

In [49]:
y_pred_precip_hours = []
for target, val in test_rolling_predictions.items():
    y_pred_precip_hours.extend(val['precipitation_hours'])
y_pred_precip_hours = np.array(y_pred_precip_hours)

In [50]:
y_pred_weather_code = []
for target, val in test_rolling_predictions.items():
    y_pred_weather_code.extend(val['weather_code'])
y_pred_weather_code = np.array(y_pred_weather_code)

In [51]:
mae = mean_absolute_error(y_true=y_true_rain, y_pred=y_pred_rain)
mse = mean_squared_error(y_true=y_true_rain, y_pred=y_pred_rain)
print(f"Rain Sum - MAE: {mae}, MSE: {mse}, RMSE: {np.sqrt(mse)}")

Rain Sum - MAE: 0.4027042547861735, MSE: 0.5581420469954249, RMSE: 0.7470890489061026


In [52]:
mae = mean_absolute_error(y_true=y_true_precip_hours, y_pred=y_pred_precip_hours)
mse = mean_squared_error(y_true=y_true_precip_hours, y_pred=y_pred_precip_hours)
print(f"Precip Hours - MAE: {mae}, MSE: {mse}, RMSE: {np.sqrt(mse)}")

Precip Hours - MAE: 2.870689272880554, MSE: 23.202306509093223, RMSE: 4.816877257009278


In [53]:
rp = classification_report(y_true_weather_code, y_pred_weather_code)
print(rp)

              precision    recall  f1-score   support

           0       0.80      0.50      0.62         8
           1       0.43      0.75      0.55         4

    accuracy                           0.58        12
   macro avg       0.61      0.62      0.58        12
weighted avg       0.68      0.58      0.59        12

